# Lab | Pandas

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

The data includes information such as customer ID, state, gender, education, income, and other variables that can be used to perform various analyses.

Throughout the lab, we will be using the pandas library in Python to manipulate and analyze the data. Pandas is a powerful library that provides various data manipulation and analysis tools, including the ability to load and manipulate data from a variety of sources, including CSV files.

### Data Description

- Customer - Customer ID

- ST - State where customers live

- Gender - Gender of the customer

- Education - Background education of customers 

- Customer Lifetime Value - Customer lifetime value(CLV) is the total revenue the client will derive from their entire relationship with a customer. In other words, is the predicted or calculated value of a customer over their entire duration as a policyholder with the insurance company. It is an estimation of the net profit that the insurance company expects to generate from a customer throughout their relationship with the company. Customer Lifetime Value takes into account factors such as the duration of the customer's policy, premium payments, claim history, renewal likelihood, and potential additional services or products the customer may purchase. It helps insurers assess the long-term profitability and value associated with retaining a particular customer.

- Income - Customers income

- Monthly Premium Auto - Amount of money the customer pays on a monthly basis as a premium for their auto insurance coverage. It represents the recurring cost that the insured person must pay to maintain their insurance policy and receive coverage for potential damages, accidents, or other covered events related to their vehicle.

- Number of Open Complaints - Number of complaints the customer opened

- Policy Type - There are three type of policies in car insurance (Corporate Auto, Personal Auto, and Special Auto)

- Vehicle Class - Type of vehicle classes that customers have Two-Door Car, Four-Door Car SUV, Luxury SUV, Sports Car, and Luxury Car

- Total Claim Amount - the sum of all claims made by the customer. It represents the total monetary value of all approved claims for incidents such as accidents, theft, vandalism, or other covered events.


External Resources: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

## Challenge 1: Understanding the data

In this challenge, you will use pandas to explore a given dataset. Your task is to gain a deep understanding of the data by analyzing its characteristics, dimensions, and statistical properties.

- Identify the dimensions of the dataset by determining the number of rows and columns it contains.
- Determine the data types of each column and evaluate whether they are appropriate for the nature of the variable. You should also provide suggestions for fixing any incorrect data types.
- Identify the number of unique values for each column and determine which columns appear to be categorical. You should also describe the unique values of each categorical column and the range of values for numerical columns, and give your insights.
- Compute summary statistics such as mean, median, mode, standard deviation, and quartiles to understand the central tendency and distribution of the data for numerical columns. You should also provide your conclusions based on these summary statistics.
- Compute summary statistics for categorical columns and providing your conclusions based on these statistics.

### Paso 0. Cargar los datos

`pd.read_csv()` acepta directamente una URL, así que no hace falta descargar el archivo. `head()` enseña las 5 primeras filas para ver qué pinta tienen los datos.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)
df.head()

,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


### 1.1 Dimensiones

`df.shape` devuelve una tupla `(filas, columnas)`.

Al mirar el final del DataFrame con `tail()` se ve que hay muchas filas **completamente vacías**. Las cuento con `df.isna().all(axis=1)`: `isna()` marca cada celda vacía, y `.all(axis=1)` dice, fila a fila, si **todas** sus celdas lo están.

In [2]:
rows, columns = df.shape
print(f"Rows: {rows}, Columns: {columns}")

empty_rows = df.isna().all(axis=1).sum()
print(f"Completely empty rows: {empty_rows}")
df.tail()

Rows: 4008, Columns: 11
Completely empty rows: 2937


,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
4003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Decisión:** quito las filas totalmente vacías con `dropna(how="all")`. No tienen ningún dato, así que no aportan nada al análisis y solo inflan el número de filas. `how="all"` solo borra una fila si **todas** sus columnas están vacías; con el valor por defecto (`how="any"`) borraría también clientes a los que les falta un único dato, como el género, y eso sería perder información.

Tras quitarlas, el dataset real tiene 1071 clientes y 11 columnas. El tratamiento del resto de nulos se hace en el lab de limpieza.

In [3]:
df = df.dropna(how="all")
print(f"Rows after removing empty rows: {df.shape[0]}")

Rows after removing empty rows: 1071


### 1.2 Tipos de datos

`df.dtypes` muestra el tipo de cada columna. `object` es el tipo que pandas usa para texto.

In [4]:
df.dtypes

Customer                      object
ST                            object
GENDER                        object
Education                     object
Customer Lifetime Value       object
Income                       float64
Monthly Premium Auto         float64
Number of Open Complaints     object
Policy Type                   object
Vehicle Class                 object
Total Claim Amount           float64
dtype: object

**Evaluación de los tipos:**

| Columna | Tipo actual | ¿Correcto? | Propuesta |
|---|---|---|---|
| Customer, ST, GENDER, Education, Policy Type, Vehicle Class | object | sí, son texto | las que tienen pocos valores distintos se podrían pasar a `category` para ahorrar memoria |
| Customer Lifetime Value | object | **no** | es un número, pero viene con `%` al final (`697953.59%`). Hay que quitar el `%` y convertir a `float` |
| Income, Monthly Premium Auto, Total Claim Amount | float64 | sí | Income y Monthly Premium Auto no tienen decimales, podrían ser `int` cuando no haya nulos |
| Number of Open Complaints | object | **no** | viene como `1/0/00`; el número de quejas es el valor del medio. Hay que extraerlo con `split("/")` y pasarlo a `int` |

### 1.3 Valores únicos y columnas categóricas

`nunique()` cuenta los valores distintos de cada columna. Una columna con pocos valores distintos respecto al número de filas suele ser **categórica**.

In [5]:
df.nunique()

Customer                     1071
ST                              8
GENDER                          5
Education                       6
Customer Lifetime Value      1027
Income                        774
Monthly Premium Auto          132
Number of Open Complaints       6
Policy Type                     3
Vehicle Class                   6
Total Claim Amount            761
dtype: int64

**Categóricas:** `ST` (8), `GENDER` (5), `Education` (6), `Number of Open Complaints` (6), `Policy Type` (3) y `Vehicle Class` (6).

`Customer` tiene un valor distinto por fila, es un identificador, no una categoría. `Customer Lifetime Value`, `Income`, `Monthly Premium Auto` y `Total Claim Amount` tienen cientos de valores distintos: son numéricas.

Ahora veo los valores de cada categórica con un bucle, y el rango (mínimo y máximo) de las numéricas.

In [6]:
categorical_columns = ["ST", "GENDER", "Education", "Number of Open Complaints", "Policy Type", "Vehicle Class"]

for column in categorical_columns:
    print(f"{column}: {df[column].unique()}")

ST: ['Washington' 'Arizona' 'Nevada' 'California' 'Oregon' 'Cali' 'AZ' 'WA']
GENDER: [nan 'F' 'M' 'Femal' 'Male' 'female']
Education: ['Master' 'Bachelor' 'High School or Below' 'College' 'Bachelors' 'Doctor']
Number of Open Complaints: ['1/0/00' '1/2/00' '1/1/00' '1/3/00' '1/5/00' '1/4/00']
Policy Type: ['Personal Auto' 'Corporate Auto' 'Special Auto']
Vehicle Class: ['Four-Door Car' 'Two-Door Car' 'SUV' 'Luxury SUV' 'Sports Car'
 'Luxury Car']


In [7]:
numerical_columns = df.select_dtypes(include="number").columns

for column in numerical_columns:
    print(f"{column}: from {df[column].min()} to {df[column].max()}")

Income: from 0.0 to 99960.0
Monthly Premium Auto: from 61.0 to 35354.0
Total Claim Amount: from 0.382107 to 2893.239678


**Conclusiones sobre los valores:**

- **ST** mezcla nombres completos y abreviaturas del mismo estado: `Cali` y `California`, `WA` y `Washington`, `AZ` y `Arizona`. Hay 8 valores pero en realidad son 5 estados.
- **GENDER** tiene 5 formas de decir 2 cosas: `F`, `M`, `Male`, `female`, `Femal`.
- **Education** tiene `Bachelor` y `Bachelors`, que son lo mismo.
- **Number of Open Complaints** tiene el formato raro `1/x/00` ya comentado.
- **Income** va de 0 a casi 100 000. Hay muchos clientes con ingreso 0 (probablemente sin empleo).
- **Monthly Premium Auto** va de 61 a 35 354. Un seguro de coche de 35 000 al mes no es realista: son valores atípicos o errores de captura.
- **Total Claim Amount** va de menos de 1 a casi 2900.

### 1.4 Estadísticas de las columnas numéricas

- `describe()` da de una vez la media (`mean`), la desviación típica (`std`), el mínimo, los cuartiles (25 %, 50 %, 75 %) y el máximo.
- La mediana es el 50 % de `describe()`, pero la calculo aparte con `median()` para tenerla junto a la moda.
- `mode()` devuelve un DataFrame (puede haber empates), por eso me quedo con la primera fila con `.iloc[0]`.

In [8]:
df.describe()

,Income,Monthly Premium Auto,Total Claim Amount
count,1071.000000,1071.000000,1071.000000
mean,39295.701214,193.234360,404.986909
std,30469.427060,1601.190369,293.027260
min,0.000000,61.000000,0.382107
25%,14072.000000,68.000000,202.157702
50%,36234.000000,83.000000,354.729129
75%,64631.000000,109.500000,532.800000
max,99960.000000,35354.000000,2893.239678


In [9]:
summary = pd.DataFrame({
    "mean": df[numerical_columns].mean(),
    "median": df[numerical_columns].median(),
    "mode": df[numerical_columns].mode().iloc[0],
    "std": df[numerical_columns].std(),
})
summary.round(2)

,mean,median,mode,std
Income,39295.70,36234.00,0.0,30469.43
Monthly Premium Auto,193.23,83.00,65.0,1601.19
Total Claim Amount,404.99,354.73,321.6,293.03


In [10]:
zero_income = (df["Income"] == 0).sum()
print(f"Customers with income 0: {zero_income} ({zero_income / len(df):.1%})")
print("Highest monthly premiums:", sorted(df["Monthly Premium Auto"])[-8:])

Customers with income 0: 252 (23.5%)
Highest monthly premiums: [1005.0, 6464.0, 6464.0, 6464.0, 6464.0, 10202.0, 35353.0, 35354.0]


**Conclusiones:**

- **Income:** media ≈ 39 300 y mediana ≈ 36 200, bastante parecidas, así que la distribución es bastante simétrica. La moda es 0: casi 1 de cada 4 clientes (252, un 23,5 %) declara ingreso 0. Es un grupo importante que conviene analizar aparte.
- **Monthly Premium Auto:** la mediana es 83 y el 75 % de los clientes paga 109,5 o menos, pero la media sube a 193 y la desviación típica a 1601. La causa son unos pocos valores enormes (6464, 10 202, 35 353...). La **media no es representativa** aquí; la mediana describe mucho mejor al cliente típico.
- **Total Claim Amount:** media 405 y mediana 355. La media es mayor que la mediana, señal de asimetría a la derecha: la mayoría reclama cantidades moderadas y unos pocos clientes reclaman mucho (máximo ≈ 2893).

### 1.5 Estadísticas de las columnas categóricas

`describe(include="object")` da para cada columna de texto: cuántos valores no nulos hay (`count`), cuántos distintos (`unique`), el más frecuente (`top`) y cuántas veces aparece (`freq`). Para ver proporciones uso `value_counts(normalize=True)`, que devuelve el porcentaje en lugar del número.

In [11]:
df.describe(include="object")

,Customer,ST,GENDER,Education,Customer Lifetime Value,Number of Open Complaints,Policy Type,Vehicle Class
count,1071,1071,954,1071,1068,1071,1071,1071
unique,1071,8,5,6,1027,6,3,6
top,RB50392,Oregon,F,Bachelor,445811.34%,1/0/00,Personal Auto,Four-Door Car
freq,1,320,457,324,4,830,780,576


In [12]:
for column in ["ST", "GENDER", "Education", "Policy Type", "Vehicle Class"]:
    print(df[column].value_counts(normalize=True).round(3), "\n")

ST
Oregon        0.299
California    0.197
Arizona       0.174
Cali          0.112
Nevada        0.092
Washington    0.076
WA            0.028
AZ            0.023
Name: proportion, dtype: float64 

GENDER
F         0.479
M         0.433
Male      0.041
female    0.029
Femal     0.018
Name: proportion, dtype: float64 

Education
Bachelor                0.303
College                 0.292
High School or Below    0.276
Master                  0.088
Doctor                  0.035
Bachelors               0.007
Name: proportion, dtype: float64 

Policy Type
Personal Auto     0.728
Corporate Auto    0.218
Special Auto      0.053
Name: proportion, dtype: float64 

Vehicle Class
Four-Door Car    0.538
Two-Door Car     0.191
SUV              0.186
Sports Car       0.053
Luxury SUV       0.019
Luxury Car       0.013
Name: proportion, dtype: float64 



**Conclusiones:**

- **Estado:** con los datos tal como vienen, Oregon es el más frecuente (320 clientes, ≈ 30 %). Pero si sumamos `California` y `Cali` (211 + 120 = 331), California pasa a ser el primero. Con los datos sin limpiar, la foto engaña.
- **Género:** es la única columna categórica con nulos (954 valores de 1071). Sumando las variantes, hay algo más de mujeres que de hombres.
- **Educación:** Bachelor, College y High School or Below están muy igualados (≈ 30 % cada uno); Master y Doctor son minoría.
- **Tipo de póliza:** Personal Auto domina claramente (≈ 73 %).
- **Vehículo:** más de la mitad son Four-Door Car; los de lujo (Sports Car, Luxury SUV, Luxury Car) son menos del 10 %.

## Challenge 2: analyzing the data

### Exercise 1

The marketing team wants to know the top 5 less common customer locations. Create a pandas Series object that contains the customer locations and their frequencies, and then retrieve the top 5 less common locations in ascending order.

**Qué hago:** `value_counts()` crea una Series con cada estado y su frecuencia. Por defecto viene ordenada de mayor a menor, así que para las 5 **menos** comunes en orden ascendente uso `sort_values()` y me quedo con las 5 primeras con `head(5)`.

**Alternativa equivalente:** `value_counts().nsmallest(5)`.

**Aviso:** con los datos sin limpiar, `AZ` y `WA` salen como "lugares poco comunes", pero son abreviaturas de Arizona y Washington. Después del lab de limpieza el resultado cambiará. Lo muestro también con los nombres unificados para que se vea la diferencia.

In [13]:
locations = df["ST"].value_counts()
less_common_locations = locations.sort_values().head(5)
less_common_locations

ST
AZ             25
WA             30
Washington     81
Nevada         98
Cali          120
Name: count, dtype: int64

In [14]:
state_names = {"AZ": "Arizona", "Cali": "California", "WA": "Washington"}
locations_unified = df["ST"].replace(state_names).value_counts()
locations_unified.sort_values().head(5)

ST
Nevada         98
Washington    111
Arizona       211
Oregon        320
California    331
Name: count, dtype: int64

### Exercise 2

The sales team wants to know the total number of policies sold for each type of policy. Create a pandas Series object that contains the policy types and their total number of policies sold, and then retrieve the policy type with the highest number of policies sold.

*Hint:*
- *Using value_counts() method simplifies this analysis.*
- *Futhermore, there is a method that returns the index of the maximum value in a column or row.*


**Qué hago:** `value_counts()` sobre `Policy Type` da el número de pólizas de cada tipo. `idxmax()` devuelve la **etiqueta** (el índice) del valor más alto, es decir, el nombre del tipo de póliza más vendido.

**Por qué `idxmax()` y no `max()`:** `max()` devolvería el número (780), no el nombre del tipo de póliza.

In [15]:
policies_sold = df["Policy Type"].value_counts()
print(policies_sold, "\n")

best_selling_policy = policies_sold.idxmax()
print(f"Policy type with the most policies sold: {best_selling_policy} ({policies_sold.max()} policies)")

Policy Type
Personal Auto     780
Corporate Auto    234
Special Auto       57
Name: count, dtype: int64 

Policy type with the most policies sold: Personal Auto (780 policies)


### Exercise 3

The sales team wants to know if customers with Personal Auto have a lower income than those with Corporate Auto. How does the average income compare between the two policy types?

- Use *loc* to create two dataframes: one containing only Personal Auto policies and one containing only Corporate Auto policies.
- Calculate the average income for each policy.
- Print the results.

**Qué hago:** con `loc` y una condición booleana (`df["Policy Type"] == "Personal Auto"`) me quedo solo con las filas de ese tipo de póliza. Hago lo mismo con Corporate Auto y calculo la media de `Income` de cada DataFrame.

**Por qué `loc[condición]`:** es la forma recomendada de filtrar filas en pandas; deja claro que estoy seleccionando por etiqueta o condición.

**Por qué calculo también la mediana:** en el Challenge 1 vimos que muchos clientes tienen ingreso 0. Si un grupo tiene más ceros que otro, la media se mueve mucho. Comparar también la mediana confirma si la diferencia es real.

In [16]:
personal_auto = df.loc[df["Policy Type"] == "Personal Auto"]
corporate_auto = df.loc[df["Policy Type"] == "Corporate Auto"]

personal_mean = personal_auto["Income"].mean()
corporate_mean = corporate_auto["Income"].mean()

print(f"Average income - Personal Auto: {personal_mean:.2f}")
print(f"Average income - Corporate Auto: {corporate_mean:.2f}")
print(f"Difference: {corporate_mean - personal_mean:.2f}")

print(f"\nMedian income - Personal Auto: {personal_auto['Income'].median():.2f}")
print(f"Median income - Corporate Auto: {corporate_auto['Income'].median():.2f}")

Average income - Personal Auto: 38180.70
Average income - Corporate Auto: 41390.31
Difference: 3209.61

Median income - Personal Auto: 35211.50
Median income - Corporate Auto: 39155.50


**Conclusión:** sí, los clientes de Personal Auto tienen de media un ingreso más bajo que los de Corporate Auto (≈ 38 200 frente a ≈ 41 400, unos 3200 menos). La mediana muestra la misma dirección (≈ 35 200 frente a ≈ 39 200), así que la diferencia no se debe solo a unos pocos valores extremos. Aun así es una diferencia moderada; para afirmar que es significativa haría falta una prueba estadística, que se verá más adelante en el bootcamp.

### Bonus: Exercise 4


Your goal is to identify customers with a high policy claim amount.

Instructions:

- Review again the statistics for total claim amount to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount data. 

*Note: When analyzing data, we often want to focus on certain groups of values to gain insights. Percentiles are a useful tool to help us define these groups. A percentile is a measure that tells us what percentage of values in a dataset are below a certain value. For example, the 75th percentile represents the value below which 75% of the data falls. Similarly, the 25th percentile represents the value below which 25% of the data falls. When we talk about the top 25%, we are referring to the values that fall above the 75th percentile, which represent the top quarter of the data. On the other hand, when we talk about the bottom 25%, we are referring to the values that fall below the 25th percentile, which represent the bottom quarter of the data. By focusing on these groups, we can identify patterns and trends that may be useful for making decisions and taking action.*

*Hint: look for a method that gives you the percentile or quantile 0.75 and 0.25 for a Pandas Series.*

*Hint 2: check `Boolean selection according to the values of a single column` in https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9*

**Paso 1. Repasar la variable.** `describe()` de `Total Claim Amount`.

**Paso 2. Calcular el percentil 75.** `quantile(0.75)` devuelve el valor por debajo del cual está el 75 % de los clientes.

**Paso 3. Filtrar.** Con `loc` me quedo con los clientes cuya reclamación supera ese valor: el 25 % que más reclama.

**Paso 4. Estadísticas del grupo** con `describe()`, y comparo el tipo de vehículo del grupo con el del total usando proporciones.

In [17]:
df["Total Claim Amount"].describe()

count    1071.000000
mean      404.986909
std       293.027260
min         0.382107
25%       202.157702
50%       354.729129
75%       532.800000
max      2893.239678
Name: Total Claim Amount, dtype: float64

In [18]:
claim_75 = df["Total Claim Amount"].quantile(0.75)
print(f"75th percentile of Total Claim Amount: {claim_75:.2f}")

high_claim_customers = df.loc[df["Total Claim Amount"] > claim_75]
print(f"Customers with a high claim amount: {len(high_claim_customers)}")
high_claim_customers.head()

75th percentile of Total Claim Amount: 532.80
Customers with a high claim amount: 264


,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
17,OE15005,Cali,NaN,College,394524.16%,28855.0,101.0,1/0/00,Personal Auto,SUV,647.442031
23,TZ98966,Nevada,NaN,Bachelor,245019.10%,0.0,73.0,1/3/00,Corporate Auto,Four-Door Car,554.376763
26,US89481,California,NaN,Bachelor,394637.21%,0.0,111.0,1/0/00,Personal Auto,Four-Door Car,799.200000


In [19]:
high_claim_customers.describe()

,Income,Monthly Premium Auto,Total Claim Amount
count,264.000000,264.000000,264.000000
mean,23677.344697,165.193182,782.228263
std,27013.483721,623.930992,292.751640
min,0.000000,63.000000,537.600000
25%,0.000000,99.000000,606.521741
50%,18807.000000,114.000000,679.597985
75%,42423.750000,133.250000,851.400000
max,99316.000000,10202.000000,2893.239678


In [20]:
comparison = pd.DataFrame({
    "high_claim": high_claim_customers["Vehicle Class"].value_counts(normalize=True),
    "all_customers": df["Vehicle Class"].value_counts(normalize=True),
}).round(3)
comparison

,high_claim,all_customers
Vehicle Class,,
Four-Door Car,0.261,0.538
Luxury Car,0.042,0.013
Luxury SUV,0.072,0.019
SUV,0.383,0.186
Sports Car,0.117,0.053
Two-Door Car,0.125,0.191


**Conclusiones:**

- El umbral del 25 % superior es ≈ 533. Por encima hay 264 clientes, con una reclamación media de ≈ 782 (el doble de la media general, ≈ 405).
- Este grupo tiene un **ingreso mucho más bajo**: media ≈ 23 700 frente a ≈ 39 300 en el total, y al menos una cuarta parte tiene ingreso 0 (el percentil 25 es 0).
- Los **SUV** pesan mucho más en este grupo (≈ 38 % frente a ≈ 19 % en el total), y los deportivos y de lujo también aparecen más. El Four-Door Car, que es el más común, está infrarrepresentado.
- Idea para el negocio: revisar el precio de las pólizas de SUV y vehículos de lujo, y seguir de cerca a los clientes con ingreso 0, que concentran reclamaciones altas.